# Lecture 1: Introduction to Applied Statistics

## Consequential decisions with data

A hospital gets fined \$500K for "too many" readmissions — but does it actually have a quality problem, or does it just serve sicker patients? An Airbnb host wonders if they're underpricing by \$50/night. A pharmaceutical company must decide whether a \$2 billion drug actually works.

These are kind of questions that drive this course. 
Each one involves real data, a model that could be wrong, 
and a decision with a major impact.

Here are ten examples we'll return to throughout the quarter:

| Question | Data | Decision |
|----------|------|----------|
| Hospital readmission penalties | CMS readmission rates by hospital | Which hospitals should be fined? |
| Harden's mid-range shot | NBA shot charts (location, outcome) | Should he stop shooting mid-range twos? |
| Wealthfront tax-loss harvesting | Portfolio returns, covariance matrices | Which lots to sell today to save on taxes? |
| WFP food allocation in Yemen | Hunger indicators from surveys and satellites | How to feed 2M more people at the same cost? |
| NextEra solar farm siting | 30 years of hourly solar irradiance (NREL) | Which parcels maximize energy per dollar? |
| Pfizer vaccine efficacy | Randomized trial: 8 vs. 162 cases in 43K patients | Enough evidence for emergency authorization? |
| NC gerrymandering | Precinct-level votes + 24K simulated district maps | Was the 10–3 seat split geographic luck or manipulation? |
| Zillow's iBuying loss | Home prices, Zestimate predictions | Why did the algorithm overpay on 9,790 homes? |
| COMPAS bail scores | Risk scores and recidivism outcomes by race | Why is the false positive rate 45% for Black defendants vs. 23% for white? |
| Netflix recommendations | 100M ratings from 480K users on 17K movies | Which movies to recommend to which users? |

: {.striped .hover tbl-colwidths="[30,35,35]"}

**How do you make these decisions with data?**
We'll spend the quarter building the tools to answer these questions. 

> *"In God we trust; all others must bring data."* — W. Edwards Deming


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['font.size'] = 12

# Load data
DATA_DIR = 'https://raw.githubusercontent.com/stanford-mse-125/book/main/data'

## What makes a decision consequential?

A decision is **consequential** when getting it wrong has serious impact — and someone has to live with the outcome. The examples above aren't just interesting data problems; they're decisions where the cost of being wrong is high.

**Financial impact.** Zillow's iBuying algorithm overpaid on nearly every home it purchased in Q3 2021, leading to an \$881M write-down and the shutdown of an entire business unit. On the other side, Wealthfront manages \$50B in assets — a bug in its tax-loss harvesting algorithm costs real clients real money.

**Human impact.** CMS hospital readmission penalties can cost a hospital \$500K–\$1M per year, directly affecting the resources available for patient care. COMPAS bail scores determine who walks free and who waits in jail — and as we'll see in Act 3, the algorithm's errors fall unevenly across racial groups in a way that no single fix can resolve. The Pfizer vaccine trial — 8 cases vs. 162 in 43,000 patients — determined whether millions of people got vaccinated.

**Irreversibility.** Some decisions can't be undone. A prison sentence served on a false positive can't be returned. A candidate elected based on gerrymandered districts can't be un-elected. Surgical decisions, infrastructure investments, closing a business unit — these are one-way doors.

:::{.callout-note}
## The fairness impossibility
The COMPAS controversy reveals a deep mathematical constraint, not just a software bug. ProPublica found that Black defendants who did *not* reoffend were nearly twice as likely to be flagged high-risk (false positive rate: 45% vs. 23%). Northpointe, the algorithm's maker, countered that among defendants *scored* as high-risk, the actual recidivism rate was similar across races (~60%). Both were correct — but they measured fairness differently.

It turns out that when two groups have different base rates of reoffending, it is *mathematically impossible* to equalize both false positive rates and predictive values at the same time (Chouldechova 2017; Kleinberg, Mullainathan, and Raghavan 2016). This isn't a bug to fix — it's a tradeoff to navigate. Which fairness criterion to prioritize is a values question, not a statistical one. We'll formalize this in Act 3.
:::

**Scale.** The World Food Programme's reallocation model affects 2 million people simultaneously. Netflix's recommendation algorithm shapes what 200 million users watch. When a decision affects many people at once, even small errors can be consequential.

:::{.callout-tip}
## Think About It
Look back at the ten questions in the table above. Pick three from different domains. Which of these four dimensions (financial, human, irreversible, scale) applies to each? Most consequential decisions hit more than one.
:::

MS&E graduates will make exactly these kinds of decisions. 
This course is designed to teach you the tools — building models, quantifying uncertainty, reasoning about causation — to make these decisions with data.
By the end of the quarter, you'll have a toolkit to approach any data-driven decision with rigor and confidence.

## What is applied statistics?

**Applied statistics** is the science of making decisions under uncertainty using data. You'll learn to work at the intersection of three disciplines:

- **Probability** (from MS&E 120) — the mathematics of uncertainty
- **Computing** — the tools for wrangling real datasets
- **Domain knowledge** — the context that turns numbers into insight

As John Tukey put it: *"The best thing about being a statistician is that you get to play in everyone's backyard."* The same tools you'll learn here apply to healthcare, housing, sports, and drug development.

## Problems with messy data

Real data is never clean. Before you build a model or run a test, you need to understand three ways data goes wrong:

**Noisy data.** Some values suffer errors, inaccuracies, or even malicious corruption. A sensor drifts out of calibration. A human mistypes a zip code. An adversary submits fake reviews. The values are present but wrong — and the damage can be subtle. A single decimal-point error in a drug dosage field could flip a clinical conclusion.

**Missing data.** Some values are not recorded, suppressed, lost, or inconsistent. We saw this in the hospital data: "Too Few to Report" means small hospitals lack enough cases, so their readmission numbers are systematically absent. Missing data is not just a gap to fill — it's a signal about the data-generating process.

**Heterogeneous data.** Real datasets mix many types of values: continuous measurements (temperature, price), discrete counts (readmissions), nominal categories (neighborhood, diagnosis), ordinal rankings (star ratings), free text (reviews), networks (social graphs), and images. Most statistical methods assume one type. Working with real data means choosing how to represent — or discard — each type.

:::{.callout-note}
## Informative vs. uninformative missingness
Not all missing data is created equal. Consider two examples:

- **Ambulance heart-rate monitor:** The reading is missing because the patient was too critical to hook up — the missingness itself predicts the outcome. This is **informative missingness** (also called *missing not at random*, or MNAR).
- **Weather station:** The reading is missing because the battery died — the gap has nothing to do with the weather. This is **uninformative missingness** (also called *missing completely at random*, or MCAR).

The distinction matters enormously. If you drop rows with informative missingness, you bias your analysis — the missing cases are systematically different from the observed ones. We'll return to this idea throughout the course.
:::

:::{.callout-warning}
## Outliers distort everything
An **outlier** is an observation far from the bulk of the data. Sometimes it's a genuine extreme value (a $10,000/night penthouse on Airbnb); sometimes it's a data error (a price of -$1). Either way, outliers can dominate summary statistics like the mean and standard deviation. Robust alternatives — the median, trimmed means, the interquartile range — resist their influence. Always plot the distribution before trusting a summary number.
:::

## The three acts of this course

The course follows a three-act structure. Each act builds on the last:

**Act 1: Build Models** (Lectures 1–7) — Explore data, clean it, and build predictive models. We'll use regression, feature engineering, and decision trees on the Airbnb and hospital datasets.

**Act 2: Trust Models** (Lectures 8–12) — Sampling, hypothesis testing, and regression inference. We'll ask: how precise are our estimates? Is the drug effect real? Which coefficients matter?

**Act 3: See Further** (Lectures 13–19) — Classification, PCA, clustering, time series, tree-based methods, and causal inference. We'll move from "what happened" to "why."

We'll see the ten questions from the opening recur throughout the course:

| Question | Topics | Act |
|----------|--------|-----|
| Hospital readmission penalties | EDA, hypothesis testing | I → II |
| Harden's mid-range shot | EDA, conditional expected value | I |
| Wealthfront tax-loss harvesting | Optimization, regression | I |
| WFP food allocation | Linear algebra, optimization | I |
| NextEra solar farm siting | Feature engineering, regression | I |
| Pfizer vaccine efficacy | Hypothesis testing, multiple testing | II |
| NC gerrymandering | Permutation tests, simulation | II |
| Zillow's iBuying algorithm | Regression, prediction intervals, backtesting | I → II |
| COMPAS bail scores | Classification, fairness | III |
| Netflix recommendations | PCA, SVD, matrix completion | III |

: {.striped .hover tbl-colwidths="[30,40,30]"}

## A first look: hospital readmissions

Let's start with a real dataset. The Centers for Medicare & Medicaid Services (CMS) tracks how often patients are readmitted to hospitals within 30 days of discharge. Hospitals with "too many" readmissions get fined — up to 3% of their Medicare payments, which translates to \$500K to \$1M per year for a large hospital.

In [ ]:
# Load hospital readmissions data
readmissions = pd.read_csv(f'{DATA_DIR}/hospital-readmissions/hrrp_full.csv')
print(f"Shape: {readmissions.shape[0]:,} rows x {readmissions.shape[1]} columns")
readmissions.head(10)

Each row is one hospital-condition pair: a hospital's readmission performance for a specific condition (heart attack, pneumonia, heart failure, etc.). Tables like this — rectangular grids of rows and columns — are the fundamental data structure in data science. Nearly every dataset you'll encounter in this course lives in a table (or *DataFrame*, in pandas terminology). Let's see what conditions are tracked.

In [ ]:
readmissions['Measure Name'].value_counts()

:::{.callout-important}
## Definition: Excess Readmission Ratio (ERR)
The **Excess Readmission Ratio (ERR)** is the ratio of a hospital's predicted readmission number to its expected number, after adjusting for patient risk. Above 1.0 means more readmissions than expected.
:::

In [ ]:
# Distribution of Excess Readmission Ratios
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(readmissions['Excess Readmission Ratio'].dropna(), bins=50, ax=ax,
             edgecolor='white')
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2, label='Expected = 1.0')
ax.axvline(x=1.05, color='orange', linestyle=':', linewidth=2, label='Your hospital = 1.05')
ax.set_xlabel('Excess Readmission Ratio')
ax.set_ylabel('Count')
ax.set_title('Hospital Readmission Performance Across the U.S.')
ax.legend()
plt.tight_layout()
plt.show()

Notice the distribution is centered near 1.0 — hospitals with ratios above 1.0 have more readmissions than expected, and those below have fewer. But there's real spread.

:::{.callout-tip}
## Think About It
If you ran a hospital and saw your ratio was 1.05 (the orange line), would you panic? How do you know if that's bad luck or a real problem? That's a statistics question.
:::

## Another dataset: Airbnb pricing

Now a completely different question. You're an Airbnb host in New York City. You want to set your price. Too high and nobody books; too low and you leave money on the table. What's the right price?

In [ ]:
# Load Airbnb data (just a few key columns for now)
airbnb = pd.read_csv(f'{DATA_DIR}/airbnb/listings.csv', low_memory=False,
                     usecols=['name', 'neighbourhood_group_cleansed', 'room_type',
                              'price', 'bedrooms', 'number_of_reviews'])

# Clean price column (may contain $ and commas in raw Airbnb data)
airbnb['price'] = airbnb['price'].astype(str).str.replace('[$,]', '', regex=True).astype(float)
print(f"{airbnb.shape[0]:,} listings in NYC")
airbnb.head()

In [ ]:
airbnb['price'].describe()

Look at the output above: the mean and the max. And there are listings near \$0. Already the data is telling us something: the "average" might not be very meaningful here. Extreme values — \$0 listings that aren't real prices, and sky-high outliers — distort the mean. *Outliers distort averages* — that's a theme we'll return to all quarter.

In [ ]:
# Price distribution — notice the long right tail
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(airbnb['price'].dropna(), bins=100, ax=ax, edgecolor='white')
ax.set_xlabel('Price per night ($)')
ax.set_ylabel('Count')
ax.set_title('NYC Airbnb Price Distribution')
ax.axvline(airbnb['price'].median(), color='orange', linestyle='--', lw=2,
           label=f'Median = ${airbnb["price"].median():.0f}')
ax.axvline(airbnb['price'].mean(), color='red', linestyle='--', lw=2,
           label=f'Mean = ${airbnb["price"].mean():.0f}')
ax.set_xlim(0, 1000)
ax.legend()
plt.tight_layout()
plt.show()

Notice how far the mean (red) is pulled to the right by expensive listings. The median is a better summary here. We'll dive deep into this data in Lecture 2.

:::{.callout-tip}
## Think About It
If you had a 1-bedroom apartment in Manhattan, would you price at the mean? Why or why not? What other information would you want?
:::

## Why quick analyses go wrong

Suppose you need a fast answer. You hand the hospital dataset to an AI assistant — or to an intern, or to a colleague who skims the columns and writes a quick script. The prompt: *"Which hospitals have the worst readmission rates?"*

### What a quick analysis gets right

- **Fast summary statistics**: histograms, means, counts — all in seconds
- **Decent plots**: bar charts of top/bottom hospitals, distributions by condition
- **Code that runs**: syntactically correct pandas and matplotlib

### What a quick analysis misses

- **No skepticism about the data**: the analysis won't ask "why are 15% of the values missing?" It will silently drop those rows.
- **No domain context**: "Too Few to Report" means small and rural hospitals lack enough cases to report — dropping those rows biases the results toward large urban hospitals.
- **Plausible nonsense**: ranking hospitals by raw readmission count instead of the risk-adjusted ERR would penalize hospitals that treat sicker patients. The ranking would *look* reasonable but be *wrong*.

In [ ]:
# Here's what "Too Few to Report" looks like in the data
non_numeric = readmissions[pd.to_numeric(readmissions['Number of Readmissions'],
                                         errors='coerce').isna()]
print(f"Rows with non-numeric readmission counts: {len(non_numeric):,}")
print(f"That's {len(non_numeric)/len(readmissions)*100:.1f}% of the data")
print()
print("What values do they have?")
print(non_numeric['Number of Readmissions'].value_counts())

:::{.callout-important}
## Definition: Missing Data Mechanism
A **missing data mechanism** describes *why* data is absent — not just *that* it's absent. Here, the data isn't missing randomly; a systematic pattern drives the gaps (small hospitals don't have enough cases to report).
:::

A quick analysis would drop these rows without mentioning it. But *dropping them changes the answer* — the results would be biased toward large urban hospitals.

In [ ]:
# Which hospitals have "Too Few to Report"? Let's look at a sample.
cols_to_show = ['Facility Name', 'State', 'Measure Name', 'Number of Readmissions']
available_cols = [c for c in cols_to_show if c in non_numeric.columns]
non_numeric[available_cols].head(8)

In [ ]:
# How many hospitals are affected, by condition?
fig, ax = plt.subplots(figsize=(8, 5))
(non_numeric['Measure Name']
 .value_counts()
 .plot.barh(ax=ax, color='C3', edgecolor='white'))
ax.set_xlabel('Number of hospitals with "Too Few to Report"')
ax.set_title('Missing data is NOT random — some conditions are harder to track')
plt.tight_layout()
plt.show()

**Catching problems like these is what this course teaches.** Whether the analysis comes from an AI, a colleague, or your own first pass, the habit is the same: verify the data before trusting the conclusions.

:::{.callout-warning}
## When the model moves the market: Zillow's \$881M lesson
In 2021, Zillow's iBuying program used its Zestimate algorithm to purchase homes directly from sellers. The model said "this home is worth \$400K," so Zillow bought it. In Q3 alone, the company purchased 9,790 homes — and overpaid on nearly all of them. By November, Zillow wrote down \$881M and shut down the program.

Three statistical failures compounded:

1. **No prediction intervals.** A point estimate ("worth \$400K") is useless without a range ("... plus or minus \$30K"). If you systematically buy at the upper end of your uncertainty, you overpay on average.
2. **Distribution shift.** The model was trained on pre-2020 sales. The post-COVID housing market behaved differently — prices were volatile, supply was tight, and bidding wars were common. Historical patterns broke down.
3. **Feedback loops.** Zillow's own aggressive buying inflated prices in the neighborhoods it targeted. The model's predictions changed the data it would later learn from — a vicious cycle that made the training data less representative over time.

We'll build the tools to diagnose each of these failures: prediction intervals in Act 1, hypothesis testing for distribution shift in Act 2, and causal reasoning about feedback loops in Act 3.
:::

## Key Takeaways

- **Applied statistics is about decisions under uncertainty**, not formulas in a vacuum.
- Real data is messy: missing values, outliers, confounding variables. That mess is the point.
- Any analysis — whether from an AI, a colleague, or your own first pass — deserves skepticism. Your job is to verify.
- The course has three acts: Build Models, Trust Models, See Further. Each act builds on the last.
- Every dataset has a story. Learning to read that story — and question it — is the core skill of a statistician.

## Study guide

### Key ideas

- **Applied statistics** is the science of making decisions under uncertainty using data.
- The **Excess Readmission Ratio (ERR)** compares a hospital's readmissions to what's expected given its patient mix. Above 1.0 = more readmissions than expected.
- A **missing data mechanism** describes *why* data is absent, not just *that* it's absent. In the hospital data, "Too Few to Report" means small hospitals lack enough cases — so their data is systematically suppressed, not randomly missing.
- The **null hypothesis** (preview) is the default assumption that nothing interesting is happening (e.g., "this hospital is no different from average"). Formalized in Lectures 9–10.
- Real data is **noisy** (errors, corruption), **missing** (not recorded, suppressed), and **heterogeneous** (numbers, categories, text, networks). These are the three dimensions of messy data.
- **Informative missingness** (MNAR) means the reason data is missing is related to the missing value itself. **Uninformative missingness** (MCAR) means the gap is unrelated to the value. Dropping MNAR data biases your analysis.
- Missing data is often a signal, not just a nuisance — *why* data is missing matters as much as *that* it's missing.
- Any quick analysis — from an AI, a script, or a first pass — can produce plausible-looking results that miss critical problems in the data.
- Outliers distort averages — always look at the distribution, not just summary statistics.

### Computational tools

- `pd.read_csv()` — load a CSV file into a DataFrame
- `.head()` — peek at the first few rows
- `.describe()` — summary statistics (mean, std, min, max, quartiles)
- `.value_counts()` — count unique values in a column
- `sns.histplot()` — plot a histogram
- `pd.to_numeric(errors='coerce')` — convert to numeric, turning non-numbers into NaN

### For the quiz

- No quiz this week, but expect one every Wednesday starting next week. The quizzes are closed-book, closed-notes, and designed to test your understanding of the key ideas and tools from the lectures. They often involve interpreting code snippets, analyzing data outputs, or applying concepts to new scenarios. The best way to prepare is to review the lecture notes, understand the examples we covered, and practice with the datasets on your own.